In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

flt_factor_24_to_72 = 2.36

# id cols
list_cols_id = [
    'request_datetime',
    'accountid',
    'bitdebtor',
]

# ivs
list_cols_structure = [
    'ENG-wtd_avg',
    'ENG-bk',
    'bookvalue__app',
    'amtfinanced__app',
    'ENG-loan_to_value',
    'fltgrossmonthly__income_sum',
    'ENG-franchise',
    'rp01s__tu',
    'g232s__tu',
    'g106s__tu',
    'ENG-has_codebtor',
    'ENG-has_auto',
    'derogseverityindex__ln',
    'balmag01__tu',
    'cv15__tu',
]

# col w institutions
list_cols_inst = [
    'str_institution__tu_pmthx',
]

# targets
list_str_target = [
    'Early_Pay_Delinquency_15_60_Flag',
    'Early_Pay_Delinquency_30_90_Flag',
    'Early_Pay_Delinquency_30_180_Flag',
    'Early_Pay_Delinquency_30_360_Flag',
    'Early_Pay_Delinquency_60_720_Flag',
#     'loss_at_60',
#     'loss_at_180',
#     'loss_at_360',
    'loss_at_720',
]

# list cols
list_cols = list_cols_id + list_cols_structure + list_cols_inst + list_str_target
int_len = len(list_cols)
print(f'Importing {int_len} columns')

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

#### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

#### Get Gen 12 predictions

In [ ]:
list_cols = [
    'accountid',
    'bitdebtor',
    'request_datetime',
    'ad',
    'pd',
    'lgd',
]

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/01_gen12_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df_tmp['ecnl'] = df_tmp['pd'] * df_tmp['lgd'] * flt_factor_24_to_72
# rename
dict_rename =  {
    'ad': 'gen12_ad',
    'pd': 'gen12_pd',
    'lgd': 'gen12_lgd',
    'ecnl': 'gen12_ecnl',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on=['accountid','bitdebtor','request_datetime'],
    how='left',
)
# show
#df

#### Get Gen 13 predictions

In [ ]:
list_cols = [
    'accountid',
    'bitdebtor',
    'request_datetime',
    'pd',
    'lgd',
]

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/02_gen13_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df_tmp['ecnl'] = df_tmp['pd'] * df_tmp['lgd'] * flt_factor_24_to_72
# rename
dict_rename =  {
    'pd': 'gen13_pd',
    'lgd': 'gen13_lgd',
    'ecnl': 'gen13_ecnl',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on=['accountid','bitdebtor','request_datetime'],
    how='left',
)
# show
df

#### Get the dlv1 accounts

In [ ]:
str_filename = 'df_direct.csv'
str_uri = f's3://20250121-gen-13-model-monitoring/05_remove_direct/{str_filename}'
df_tmp = pd.read_csv(str_uri)
df_tmp['ACCOUNTID'] = df_tmp['ACCOUNTID'].astype(int)
list_int_account = list(df_tmp['ACCOUNTID'])
list_int_account = list(dict.fromkeys(list_int_account))
int_naccounts = len(list_int_account)
print(f'Number of direct accounts: {int_naccounts}')

#### Remove

In [ ]:
df['accountid'] = df['accountid'].astype(int)
df['tag'] = df['accountid'].apply(
    lambda x: 1 if x in list_int_account else 0,
)
df = df[df['tag'] == 0].copy()
df.drop('tag', axis=1, inplace=True)
# show
df

#### Make a tag at the account level if pmt hx is bad

In [ ]:
df_tmp = df.groupby('accountid', as_index=False).agg({
    'ENG-wtd_avg': 'mean',
})
df_tmp['tag'] = df_tmp['ENG-wtd_avg'].apply(
    lambda x: 1 if x < 0.5 else 0,
)
flt_mn = df_tmp['tag'].mean()
print(f'Proportion of accounts with bad pmt hx: {flt_mn:0.4f}')
df_tmp = df_tmp[df_tmp['tag'] == 1].copy()
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
list_accountid = list(df_tmp['accountid'])
# creat tag
df['bad_pmt_hx'] = df['accountid'].apply(
    lambda x: 1 if x in list_accountid else 0,
)
# show
#df

#### Get chargeoff severity

In [ ]:
df['co_at_720'] = df['loss_at_720'] / df['amtfinanced__app']
# make min 0
df['co_at_720'] = df['co_at_720'].clip(lower=0)
# show
#df

#### Make 60+ in 720 and positive net charge off

In [ ]:
df['60_plus_pos_co_at_720'] = df.apply(
    lambda x: 1 if (x['Early_Pay_Delinquency_60_720_Flag'] == 1) and (x['co_at_720'] > 0) else 0,
    axis=1,
)
# show
#df

#### Create a new column that is a list

In [ ]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)
# show
#df

#### Get the unique names for institutions

In [ ]:
list_str_inst_flat = list(df['list_institutions'].explode().dropna())
list_str_inst_flat = list(dict.fromkeys(list_str_inst_flat))
int_len = len(list_str_inst_flat)
print(f'There are {int_len} unique institutions')

#### Find any with keywords

In [ ]:
list_cols = [col for col in list_str_inst_flat if 'chime' in col.lower()]
list_cols

#### Create tag for institutions

In [ ]:
list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

# show
#df

#### Tag if there was an institution of interest

In [ ]:
df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
#df

#### Save to s3

In [ ]:
%%time

str_filename = 'df_institutions.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

#### ECNL by chime (no targets yet because we are using newer data)

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'has_inst_tag': 'max',
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
})
# gen ecnls
df_tmp['gen12_ecnl'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd'] * flt_factor_24_to_72
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
# group by has_inst_tag
df_tmp = df_tmp.groupby(by='has_inst_tag', as_index=False).agg({
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen12_ecnl': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'gen13_ecnl': 'mean',
})

# map
dict_map = {
    1: 'Yes',
    0: 'No',
}
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)

# save
str_filename = 'df_pivot_all_scores.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

#### Rate over time

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'co_at_720': 'first',
})
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp['count'] = 1
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'count': 'sum',
    'has_inst_tag': 'mean',
    'co_at_720': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['count']
a = df_tmp['co_at_720']

# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')

# second y
ax2 = ax.twinx()
ax2.set_ylabel('Total Fundings')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Total Fundings')
# legend
ax2.legend(loc='upper right')

# third y
ax3 = ax.twinx()
ax3.spines['right'].set_position(('outward', 60))  # prevent overlap
ax3.set_ylabel('Charge-Off at 24 Months')
ax3.plot(x[:17], a[:17], linestyle='--', color='red', label='Charge-Off at 24 Months')
# legend
ax3.legend(loc='lower right')

# save
str_filename = 'plt_fundings.png'
str_local_path = f'{str_dirname_output}/{str_filename}'
plt.savefig(str_local_path, bbox_inches='tight')

# show
plt.show()

#### Overlay Gen 12 AD

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'gen12_ad': 'mean',
    'co_at_720': 'first',
})
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp['count'] = 1
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'count': 'sum',
    'has_inst_tag': 'mean',
    'gen12_ad': 'mean',
    'co_at_720': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['gen12_ad']
a = df_tmp['co_at_720']

# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')

# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean Gen 12 AD')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean Gen 12 AD')
# legend
ax2.legend(loc='upper right')

# third y
ax3 = ax.twinx()
ax3.spines['right'].set_position(('outward', 60))  # prevent overlap
ax3.set_ylabel('Charge-Off at 24 Months')
ax3.plot(x[:17], a[:17], linestyle='--', color='red', label='Charge-Off at 24 Months')
# legend
ax3.legend(loc='lower right')

# show
plt.show()

#### Overlay Gen 12 ECNL

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'co_at_720': 'first',
})
df_tmp['gen12_ecnl'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd'] * flt_factor_24_to_72
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp['count'] = 1
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'count': 'sum',
    'has_inst_tag': 'mean',
    'gen12_ecnl': 'mean',
    'co_at_720': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['gen12_ecnl']
a = df_tmp['co_at_720'] * flt_factor_24_to_72

# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')

# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean Gen 12 ECNL')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean Gen 12 ECNL at 72 Months')
ax2.plot(x[:17], a[:17], linestyle='--', color='red', label='Charge-Off at 72 Months')
# legend
ax2.legend(loc='upper right')

# show
plt.show()

#### Overlay Gen 13 ECNL

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'co_at_720': 'first',
})
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp['count'] = 1
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'count': 'sum',
    'has_inst_tag': 'mean',
    'gen13_ecnl': 'mean',
    'co_at_720': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['gen13_ecnl']
a = df_tmp['co_at_720'] * flt_factor_24_to_72

# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')
# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean Gen 13 ECNL')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean Gen 13 ECNL')
ax2.plot(x[:17], a[:17], linestyle='--', color='red', label='Charge-Off at 72 Months')
# legend
ax2.legend(loc='upper right')

# show
plt.show()

#### Overlay income

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'fltgrossmonthly__income_sum': 'sum',
})
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'has_inst_tag': 'mean',
    'fltgrossmonthly__income_sum': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['fltgrossmonthly__income_sum']
# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')
# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean Income')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean Income')
# legend
ax2.legend(loc='upper right')
# show
plt.show()

#### LTV

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'ENG-loan_to_value': 'first',
})
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'has_inst_tag': 'mean',
    'ENG-loan_to_value': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['ENG-loan_to_value']
# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')
# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean LTV')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean LTV')
# legend
ax2.legend(loc='upper right')
# show
plt.show()

#### Structure pivot

In [ ]:
list_cols_dv = list_cols_structure + ['bad_pmt_hx']
df['count'] = 1
dict_tmp = {key: 'mean' for key in list_cols_dv}
dict_tmp['count'] = 'count'
df_tmp = df.groupby('has_inst_tag', as_index=False).agg(dict_tmp)

# map
dict_map = {
    1: 'Yes',
    0: 'No',
}
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)

# save locally
str_filename = 'df_pivot_all_ivs.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

#### Rm too new so we can look at targets

In [ ]:
df = df[df['request_datetime'] <= '2022-11-26'].copy()
df

#### ECNLs, Delinquency, and CO severity

In [ ]:
# ecnl at account level
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'has_inst_tag': 'max',
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'co_at_720': 'first',
    '60_plus_pos_co_at_720': 'first',
    'Early_Pay_Delinquency_60_720_Flag': 'first',
})
int_n_accounts = df_tmp.shape[0]
print(f'Accounts after subsetting: {int_n_accounts}')
# get ecnl
df_tmp['gen12_ecnl_24'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd']
df_tmp['gen13_ecnl_24'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd']
# group
df_tmp = df_tmp.groupby(by='has_inst_tag', as_index=False).agg({
    'gen12_ad': 'mean',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'gen12_ecnl_24': 'mean',
    'gen13_ecnl_24': 'mean',
    'co_at_720': 'mean',
    '60_plus_pos_co_at_720': 'mean',
    'Early_Pay_Delinquency_60_720_Flag': 'mean',
})

# convert back to Yes No
dict_map = {
    1: 'Yes',
    0: 'No',
}
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)

# suggested factors
df_tmp['gen12_suggested_factor'] = df_tmp['co_at_720'] / df_tmp['gen12_ecnl_24']
df_tmp['gen13_suggested_factor'] = df_tmp['co_at_720'] / df_tmp['gen13_ecnl_24']

# save
str_filename = 'df_pivot_subset_targets_and_predictions.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

#### Analyze targets

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'has_inst_tag': 'max',
    'Early_Pay_Delinquency_15_60_Flag': 'first',
    'Early_Pay_Delinquency_30_90_Flag': 'first',
    'Early_Pay_Delinquency_30_180_Flag': 'first',
    'Early_Pay_Delinquency_30_360_Flag': 'first',
    'Early_Pay_Delinquency_60_720_Flag': 'first',
    'co_at_720': 'first',
    '60_plus_pos_co_at_720': 'first',
    'Early_Pay_Delinquency_60_720_Flag': 'first',
})
# group by
df_tmp = df_tmp.groupby(by='has_inst_tag', as_index=False).agg({
    'Early_Pay_Delinquency_15_60_Flag': 'mean',
    'Early_Pay_Delinquency_30_90_Flag': 'mean',
    'Early_Pay_Delinquency_30_180_Flag': 'mean',
    'Early_Pay_Delinquency_30_360_Flag': 'mean',
    'Early_Pay_Delinquency_60_720_Flag': 'mean',
    'co_at_720': 'mean',
    '60_plus_pos_co_at_720': 'mean',
    'Early_Pay_Delinquency_60_720_Flag': 'mean',
})

# map
dict_map = {
    1: 'Yes',
    0: 'No',
}
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)

# save
str_filename = 'df_pivot_subset_targets.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp